**YOLOv8**, **GUI** ve **Görüntü İşlemleri** için gerekli kütüphane kurulumu

In [ ]:
!pip install -q ultralytics gradio opencv-python pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 16.1 MB/s eta 0:00:00


**GUI**: Sahip olduğu özellikler:
✔ Confidence ayarı
✔ IoU ayarı
✔ Class filtreleme
✔ FPS ölçümü
✔ Toplam tespit sayısı
✔ Çıktı indirme


In [ ]:
import gradio as gr
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
import os
import time



def run_detection(model_file, image, confidence, iou, selected_class):
    # Kullanıcı model dosyası yüklemediyse işlemi durdurur ve uyarı verir
    if model_file is None:
        return None, "❌ Lütfen bir model (.pt) yükleyin.", "", 0, None

    # Kullanıcı görüntü yüklemediyse işlemi durdurur ve uyarı verir
    if image is None:
        return None, "❌ Lütfen bir görüntü yükleyin.", "", 0, None

    try:
        # YOLO modeline ait best.pt dosyasını yükle
        model = YOLO(model_file.name)

        # Gradio görüntüyü PIL olarak oluşturur. YOLO bu görüntüyü algılayamaz
        # YOLO'nun kullanabilmesi için görüntü NumPy arraye çevrilir
        img = np.array(image)

        start_time = time.time()

        # Model ile görüntü üzerinde object detection (inference) işlemi çalıştırılır
        results = model(img, conf=confidence, iou=iou)

        end_time = time.time()
        fps = 1 / (end_time - start_time)

        # Tek bir görüntü işlendiği için ilk sonuç alınır
        result = results[0]

        # Tespit edilen nesneler için bounding box çizilmiş çıktıyı oluşturur
        annotated_img = result.plot()

        # ================= EKLENEN KISIM =================
        output_text = ""
        detection_count = 0

        if result.boxes is not None and len(result.boxes) > 0:

            class_names = model.names

            for box in result.boxes:
                cls_id = int(box.cls[0])
                conf_score = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].tolist()

                class_name = class_names[cls_id]

                if selected_class != "All" and class_name != selected_class:
                    continue

                detection_count += 1

                output_text += (
                    f"Sınıf: {class_name}\n"
                    f"Confidence: {conf_score:.2f}\n"
                    f"Koordinatlar: ({int(x1)}, {int(y1)}, {int(x2)}, {int(y2)})\n\n"
                )
        # ==================================================

        # Eğer hiçbir nesne tespit edilmediyse ekrana uyarı yazdırıllır
        if output_text == "":
            output_text = "⚠️ Herhangi bir nesne tespit edilmedi"

        else:
            output_text = f"Toplam {detection_count} nesne tespit edildi\n\n" + output_text

        output_path = "annotated_output.jpg"
        Image.fromarray(annotated_img).save(output_path)

        return annotated_img, output_text, f"{fps:.2f} FPS", detection_count, output_path

    except Exception as e:
        return None, f"❌ Hata oluştu: {str(e)}", "", 0, None




# Gradio arayüzünü tanımlar ve object detection fonksiyonunu arayüze bağlar
interface = gr.Interface(

    # Kullanıcı arayüzde butona bastığında çalışacak fonksiyon
    fn=run_detection,

    # Kullanıcıdan alınacak girdiler
    inputs=[
        # Eğitilmiş YOLO model dosyasının (best.pt) yükleneceği alan
        gr.File(label="YOLO Model Dosyası (best.pt)"),

        # Üzerinde nesne tespiti yapılacak görüntünün yükleneceği alan
        # type="pil" → Görüntü PIL formatında fonksiyona gönderilir
        gr.Image(type="pil", label="Görüntü Yükle", sources=["upload", "webcam"]),

        gr.Slider(0.0, 1.0, value=0.25, step=0.05, label="Confidence Threshold"),

        gr.Slider(0.0, 1.0, value=0.45, step=0.05, label="IoU Threshold"),

        gr.Dropdown(
    choices=[
        "All",
        "aeroplane","apple","backpack","banana","baseball bat","baseball glove",
        "bear","bed","bench","bicycle","bird","boat","book","bottle","bowl",
        "broccoli","bus","cake","car","carrot","cat","cell phone","chair",
        "clock","cow","cup","diningtable","dog","donut","elephant",
        "fire hydrant","fork","frisbee","giraffe","hair drier","handbag",
        "horse","hot dog","keyboard","kite","knife","laptop","microwave",
        "motorbike","mouse","orange","oven","oxygen tank","parking meter",
        "person","pizza","pottedplant","refrigerator","remote","sandwich",
        "scissors","sheep","sink","skateboard","skis","snowboard","sofa",
        "spoon","sports ball","stop sign","suitcase","surfboard",
        "teddy bear","tennis racket","tie","toaster","toilet",
        "toothbrush","traffic light","train","truck","tvmonitor",
        "umbrella","vase","wine glass","zebra"
    ],
    value="All",
    label="Gösterilecek Sınıf"
)
    ],

    # Kullanıcıya gösterilecek çıktılar
    outputs=[
        # Nesnelerin etrafına bounding box çizilmiş çıktı görüntüsü
        gr.Image(label="Tespit Sonucu (Bounding Box)"),

        # Her nesneye ait sınıf, confidence ve koordinat bilgilerinin yazıldığı text alanı
        gr.Textbox(label="Tespit Detayları", lines=12),

        gr.Textbox(label="Inference Süresi (FPS)"),

        gr.Number(label="Toplam Tespit"),

        gr.File(label="Çıktıyı İndir")
    ],

    # Arayüzün üst kısmında görünen başlık
    title="YOLO Object Detection GUI",

    # Arayüzün ne yaptığını açıklayan kısa bilgilendirme metni
    description=(
        "Eğitilmiş YOLO modelinizi (best.pt) ve bir görüntü yükleyerek "
        "nesne tespiti yapabilirsiniz. "
        "Çıktı olarak sınıf etiketi, confidence skoru ve bounding box koordinatları gösterilir."
    ),

    theme=gr.themes.Soft()
)

# Gradio arayüzünü çalıştırır ve tarayıcıda açar
interface.launch()

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e7e6f27e6c5263beca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
